In [85]:
!pip install bayesian-torch


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [86]:

import numpy as np
import pandas as pd
import torch
from bayesian_torch.layers.variational_layers import LinearReparameterization
from bayesian_torch.models.dnn_to_bnn import get_kl_loss
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

print("Library Versions:")
print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('torch:', torch.__version__)

Library Versions:
numpy: 2.5.1
pandas: 3.0.5
torch: 2.13.0+cpu


In [87]:

n_epochs = 100
verbose_option = True

# Regression for Naval Plant Maintenance

Load dataset

In [88]:
npm = pd.read_csv('navalplantmaintenance.csv', header=None)
npm_train, npm_test = train_test_split(npm, test_size=0.25, random_state=42)
npm_train_np = npm_train.to_numpy()
npm_test_np = npm_test.to_numpy()
x_train_np = npm_train_np[:, :16]
x_test_np = npm_test_np[:, :16]
y_train_np = npm_train_np[:, 17]
y_test_np = npm_test_np[:, 17]
x_mu = x_train_np.mean(axis=0)
x_sigma = x_train_np.std(axis=0)
y_mu = y_train_np.mean(axis=0)
y_sigma = y_train_np.std(axis=0)


def scale(x, x_mu, x_sigma):
    x_sigma += 1e-16  #This is to deal with constant or near-constant columns
    return (x - x_mu) / x_sigma


def unscale(x, x_mu, x_sigma):
    x_sigma += 1e-16  #This is to deal with constant or near-constant columns
    return x_sigma * x + x_mu


x_train_np_z = scale(x_train_np, x_mu, x_sigma)
y_train_np_z = scale(y_train_np, y_mu, y_sigma)
x_test_np_z = scale(x_test_np, x_mu, x_sigma)
y_test_np_z = scale(y_test_np, y_mu, y_sigma)
x_train_t_z = torch.FloatTensor(x_train_np_z)
y_train_t_z = torch.FloatTensor(y_train_np_z)
x_test_t_z = torch.FloatTensor(x_test_np_z)
y_test_t_z = torch.FloatTensor(y_test_np_z)

1. Using PyTorch, perform variational inference using a mean-field Gaussian variational distribution and reparamaterization layers for Gaussian heteroscedastic regression.

In [89]:
def nlls(y, mu, std):
    return torch.square(y - mu) / (2.0 * torch.square(std)) + torch.log(std)


class nn(torch.nn.Module):
    def __init__(self, inputSize, hiddenSize, outputSize):
        super(nn, self).__init__()
        self.layer1 = LinearReparameterization(inputSize,
                                               hiddenSize)  # Call the PyTorch Linear Local Reparameterization layer constructor going from inputSize to hiddenSize
        self.layer2 = LinearReparameterization(hiddenSize,
                                               hiddenSize)  # Call the PyTorch Linear Local Reparameterization layer constructor going from hiddenSize to hiddenSize
        self.linear_mu = LinearReparameterization(hiddenSize,
                                                  outputSize)  # Call the PyTorch Linear Local Reparameterization layer constructor going from hiddenSize to outputSize
        self.linear_sigma = LinearReparameterization(hiddenSize,
                                                     outputSize)  # Call the PyTorch Linear Local Reparameterization layer constructor going from hiddenSize to outputSize

    def forward(self, x):
        h1 = torch.nn.functional.relu(self.layer1(x, return_kl=False))
        h2 = torch.nn.functional.relu(self.layer2(h1, return_kl=False))
        mu = self.linear_mu(h2, return_kl=False)
        sigma = torch.nn.functional.softplus(self.linear_sigma(h2, return_kl=False))
        return mu, sigma

In [90]:
x_train_t_z

tensor([[-1.5340e+00, -1.5500e+00, -1.1291e+00,  ..., -9.4888e-01,
         -1.5078e-01, -9.5088e-01],
        [-1.5340e+00, -1.5500e+00, -1.0912e+00,  ..., -9.4888e-01,
         -1.3044e+00, -1.0119e+00],
        [ 7.5376e-01,  7.7377e-01,  5.2904e-01,  ...,  9.4549e-01,
          3.8730e-01,  4.0011e-01],
        ...,
        [ 1.5722e+00,  1.5484e+00,  2.0532e+00,  ...,  1.8927e+00,
          2.2106e+00,  2.2395e+00],
        [ 3.8452e-01,  3.8648e-01,  1.1418e-01,  ..., -1.6933e-03,
          3.1161e-02,  3.7743e-02],
        [ 1.1573e+00,  1.1611e+00,  1.0702e+00,  ...,  9.4549e-01,
          1.0609e+00,  1.0776e+00]])

In [91]:
model = nn(x_train_np_z.shape[1], 25, 1)

optimizer = torch.optim.Adam(params=model.parameters(), lr=1e-3)

for i in range(n_epochs):
    mu, s = model(x_train_t_z)
    M = mu.shape[0]
    nll_loss = nlls(y_train_t_z, mu, s).mean()
    kl = get_kl_loss(model)
    loss = nll_loss + (1 / x_train_np.shape[0]) * kl  # The VI loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if verbose_option: print(i, loss)

0 tensor(0.7396, grad_fn=<AddBackward0>)
1 tensor(0.7664, grad_fn=<AddBackward0>)
2 tensor(0.6851, grad_fn=<AddBackward0>)
3 tensor(0.7083, grad_fn=<AddBackward0>)
4 tensor(0.7712, grad_fn=<AddBackward0>)
5 tensor(0.7699, grad_fn=<AddBackward0>)
6 tensor(0.8312, grad_fn=<AddBackward0>)
7 tensor(0.6845, grad_fn=<AddBackward0>)
8 tensor(0.7431, grad_fn=<AddBackward0>)
9 tensor(0.6769, grad_fn=<AddBackward0>)
10 tensor(0.6571, grad_fn=<AddBackward0>)
11 tensor(0.7731, grad_fn=<AddBackward0>)
12 tensor(0.6683, grad_fn=<AddBackward0>)
13 tensor(0.7019, grad_fn=<AddBackward0>)
14 tensor(0.6927, grad_fn=<AddBackward0>)
15 tensor(0.7585, grad_fn=<AddBackward0>)
16 tensor(0.7287, grad_fn=<AddBackward0>)
17 tensor(0.6614, grad_fn=<AddBackward0>)
18 tensor(0.6286, grad_fn=<AddBackward0>)
19 tensor(0.6722, grad_fn=<AddBackward0>)
20 tensor(0.6482, grad_fn=<AddBackward0>)
21 tensor(0.6243, grad_fn=<AddBackward0>)
22 tensor(0.6249, grad_fn=<AddBackward0>)
23 tensor(0.6277, grad_fn=<AddBackward0>)
24

2.  Compute the mean and standard deviation predictions for 20 MC sampled models

In [92]:
mc_samples = 20
n_test_examples = y_test_np.shape[0]
y_test_mus_z = np.zeros([mc_samples, n_test_examples, 1])
y_test_sigmas_z = np.zeros([mc_samples, n_test_examples, 1])
for i in range(mc_samples):
    mu, s = model(x_test_t_z)  # Get prediction for one sampled parameter vector
    y_test_mus_z[
        i] = mu.detach().numpy()  # Get the predicted probabilities of the test examples given the sampled parameter vector - this comment doesn't make sense when looking at the code, so I am doing what the code seems to ask for.
    y_test_sigmas_z[
        i] = s.detach().numpy()  # Get the entropy of the predicted probability distributions of the test examples given the sampled parameter vector. This comment doesn't make sense when looking at the code, so I am doing what the code seems to ask for.

In [93]:
y_test_mus_z

array([[[ 0.23166245],
        [ 0.15444031],
        [ 0.04169201],
        ...,
        [ 0.15211226],
        [ 0.28345564],
        [ 0.15482321]],

       [[ 0.0621912 ],
        [-0.10173159],
        [-0.07708048],
        ...,
        [-0.10142975],
        [ 0.10103726],
        [-0.10203012]],

       [[ 0.13906689],
        [ 0.04727527],
        [-0.00464082],
        ...,
        [ 0.04460192],
        [ 0.21218519],
        [ 0.0474246 ]],

       ...,

       [[ 0.04411973],
        [-0.01331594],
        [-0.04403089],
        ...,
        [-0.01360309],
        [ 0.09128985],
        [-0.01306257]],

       [[-0.06187235],
        [-0.01377227],
        [-0.00965228],
        ...,
        [-0.01135819],
        [-0.06781732],
        [-0.01390166]],

       [[ 0.06971886],
        [ 0.31552047],
        [ 0.13093445],
        ...,
        [ 0.31204933],
        [ 0.08561748],
        [ 0.31617296]]], shape=(20, 2984, 1))

3. Compute the Mean Squared Error (MSE) for the Gaussian VI model with Local Reparameterzaton layers for the test data using 20 MC samples and Bayesian model averaging.

In [94]:
y_test_mu_z = y_test_mus_z.mean(axis=0)  # Compute the mean predictions using Bayesian model averaging
y_test_mu = unscale(y_test_mu_z, y_mu, y_sigma)
print('MSE:', mean_squared_error(y_test_np, y_test_mu))

MSE: 5.67789676415256e-05


In [95]:
y_test_mu

array([[0.98772682],
       [0.98793018],
       [0.98748059],
       ...,
       [0.9879182 ],
       [0.98791601],
       [0.98793194]], shape=(2984, 1))

4. Compute the aleatoric uncertainty for each regression test example for VI model.

In [96]:
y_test_mus = unscale(y_test_mus_z, y_mu, y_sigma)
y_test_sigmas = y_test_sigmas_z * y_sigma
y_test_aleatoric = np.mean(np.square(y_test_sigmas), axis=0)  # Compute the aleatoric uncertainties
mean_prediction = np.mean(y_test_mus, axis=0)
y_test_epistemic = np.mean(np.square(y_test_mus - mean_prediction), axis=0)  # Compute the epistemic uncertainties

In [97]:
y_test_aleatoric

array([[5.83867229e-05],
       [7.52356476e-05],
       [4.41851248e-05],
       ...,
       [7.50027539e-05],
       [6.98717347e-05],
       [7.53298339e-05]], shape=(2984, 1))

In [98]:
y_test_epistemic

array([[6.92974282e-07],
       [6.57780980e-07],
       [2.58319075e-07],
       ...,
       [6.47689081e-07],
       [1.00613719e-06],
       [6.59680583e-07]], shape=(2984, 1))

# Classification for Ship Detection


Load Ship Detection Dataset

In [99]:
import torch
from torch.utils.data import Dataset, DataLoader
from bayesian_torch.models.dnn_to_bnn import get_kl_loss
from bayesian_torch.layers.flipout_layers import LinearFlipout
from torchvision.io import read_image
from torch.utils.data import random_split
from torchvision.transforms.functional import resize
import numpy as np
from pathlib import Path
import torchmetrics

ROOT_PATH = "shipsnet"
LR = 1e-4
IMG_SIZE = [80]

tensor_size = IMG_SIZE[0] ** 2 * 3


def max_scaling(image):
    image = image / 255.0
    image = torch.Tensor(image)
    image = resize(image, size=IMG_SIZE)
    return image


def normalize_img(image):
    means = torch.Tensor([[[105.0385]], [[108.1886]], [[94.9558]]])
    stds = torch.Tensor([[[48.4294]], [[40.0104]], [[38.6445]]])
    image = torch.Tensor(image)
    image = resize(image, size=IMG_SIZE)
    image = image - means
    image = image / stds
    return image


#https://pytorch.org/tutorials/beginner/basics/data_tutorial.html
class ShipDataset(Dataset):
    def __init__(self, root_path, transform=None):
        self.root_path = Path(root_path)
        self.files = list(self.root_path.rglob("*/*"))
        self.classes = list(set([int(entry.parts[-1]) for entry in self.root_path.rglob("*") if Path(entry).is_dir()]))
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        image = read_image(str(self.files[idx]))
        label = int(self.files[idx].parts[-2])
        if self.transform:
            image = self.transform(image)
        return image, label


full_dataset = ShipDataset(ROOT_PATH, transform=normalize_img, )
n_classes = len(full_dataset.classes)
print("Length of full dataset: ", len(full_dataset), " - with ", n_classes, " classes. ")
train_dataset, test_dataset = random_split(full_dataset, [0.8, 0.2])
print("Length of training dataset: ", len(train_dataset))
print("Length of Test dataset: ", len(test_dataset))

train_dataloader = DataLoader(train_dataset, batch_size=len(train_dataset))
test_dataloader = DataLoader(test_dataset, batch_size=len(test_dataset))

criterion = torch.nn.BCELoss(reduce='mean')
accuracy = torchmetrics.classification.BinaryAccuracy()


Length of full dataset:  4000  - with  2  classes. 
Length of training dataset:  3200
Length of Test dataset:  800


C:\Users\shai1\PyCharmMiscProject\.venv\Lib\site-packages\torch\nn\modules\loss.py:48: UserWarning: size_average and reduce args will be deprecated, please use reduction='mean' instead.
  self.reduction: str = _Reduction.legacy_get_string(size_average, reduce)


5. Using PyTorch, perform variational inference using a mean-field Gaussian variational distribution and flipout layers for non-linear binary classification

In [100]:
class logistic(torch.nn.Module):
    def __init__(self, inputSize, hiddenSize, outputSize):
        super(logistic, self).__init__()
        self.layer1 = LinearFlipout(inputSize,
                                    hiddenSize)  # Call the PyTorch Linear Flipout layer constructor going from inputSize to hiddenSize
        self.layer2 = LinearFlipout(hiddenSize,
                                    hiddenSize)  # Call the PyTorch Linear Flipout layer constructor going from hiddenSize to hiddenSize
        self.layer3 = LinearFlipout(hiddenSize,
                                    hiddenSize)  # Call the PyTorch Linear Flipout layer constructor going from hiddenSize to hiddenSize
        self.layer4 = LinearFlipout(hiddenSize,
                                    hiddenSize)  # Call the PyTorch Linear Flipout layer constructor going from hiddenSize to hiddenSize
        self.p = LinearFlipout(hiddenSize,
                               outputSize)  # Call the PyTorch Linear Flipout layer constructor going from hiddenSize to outputSize

    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        x = torch.nn.functional.relu(self.layer1(x, return_kl=False))
        x = torch.nn.functional.relu(self.layer2(x, return_kl=False))
        x = torch.nn.functional.relu(self.layer3(x, return_kl=False))
        x = torch.nn.functional.relu(self.layer4(x, return_kl=False))
        x = self.p(x, return_kl=False)
        return torch.sigmoid(x)

In [101]:
model = logistic(tensor_size, 50, 1)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(n_epochs):
    for data, label in train_dataloader:
        p = model(data)
        M = p.shape[0]
        nll_loss = criterion(p.squeeze(), label * 1.0)
        kl = get_kl_loss(model)
        loss = nll_loss + (1 / data.shape[0]) * kl  # Write the VI loss
        acc = accuracy(p.squeeze(), label)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if verbose_option: print(epoch, loss, acc, end="/r")
    print(epoch)

0 tensor(0.7491, grad_fn=<AddBackward0>) tensor(0.6231)/r0
1 tensor(0.7026, grad_fn=<AddBackward0>) tensor(0.6500)/r1
2 tensor(0.6970, grad_fn=<AddBackward0>) tensor(0.6516)/r2
3 tensor(0.6872, grad_fn=<AddBackward0>) tensor(0.6522)/r3
4 tensor(0.7020, grad_fn=<AddBackward0>) tensor(0.6597)/r4
5 tensor(0.6529, grad_fn=<AddBackward0>) tensor(0.6781)/r5
6 tensor(0.6355, grad_fn=<AddBackward0>) tensor(0.6934)/r6
7 tensor(0.6337, grad_fn=<AddBackward0>) tensor(0.7019)/r7
8 tensor(0.6080, grad_fn=<AddBackward0>) tensor(0.7078)/r8
9 tensor(0.6078, grad_fn=<AddBackward0>) tensor(0.7147)/r9
10 tensor(0.6310, grad_fn=<AddBackward0>) tensor(0.7047)/r10
11 tensor(0.6322, grad_fn=<AddBackward0>) tensor(0.6891)/r11
12 tensor(0.5939, grad_fn=<AddBackward0>) tensor(0.7138)/r12
13 tensor(0.6010, grad_fn=<AddBackward0>) tensor(0.7169)/r13
14 tensor(0.5796, grad_fn=<AddBackward0>) tensor(0.7206)/r14
15 tensor(0.5533, grad_fn=<AddBackward0>) tensor(0.7444)/r15
16 tensor(0.5701, grad_fn=<AddBackward0>) te

6.  Compute the predicted probabilities and entropy predictions for 20 MC sampled models

In [102]:
mc_samples = 20
n_test_examples = len(test_dataset)
y_test_probs = np.zeros([mc_samples, n_test_examples, 1])
y_test_entropies = np.zeros([mc_samples, n_test_examples, 1])
for i in range(mc_samples):
    for data, label in test_dataloader:
        results = model(data).detach().numpy()
        y_test_probs[i] = results  # Get the predicted means for one sampled parameter vector
        y_test_entropies[i] += np.array([-(x * np.log(x) + (1 - x) * np.log(1 - x)) for x in
                                         results])  # Get the entropy of the predicted probability distributions of the test examples given the sampled parameter vector

7. Compute the Bayesian model averaging predictions for each classification test example for the VI model.

In [103]:
y_test_probs_avg = np.mean(y_test_probs, axis=0)  # Compute Bayesian model averaging predictions

In [104]:
y_test_probs_avg

array([[1.10250131e-01],
       [1.01501124e-01],
       [9.66334459e-01],
       [6.78494190e-01],
       [3.13669992e-02],
       [8.98911372e-01],
       [8.93044229e-01],
       [1.95428605e-02],
       [1.28970535e-01],
       [8.56712757e-01],
       [2.78027426e-01],
       [5.25633128e-02],
       [9.56804428e-03],
       [4.98897871e-02],
       [1.15836154e-01],
       [5.80749418e-03],
       [2.35209782e-02],
       [6.22358779e-02],
       [1.78594447e-02],
       [1.21000186e-01],
       [7.98250924e-02],
       [1.45092663e-01],
       [1.32795127e-01],
       [1.89041893e-01],
       [1.27183896e-01],
       [7.70640150e-01],
       [5.92039727e-01],
       [4.12557494e-01],
       [1.78873887e-01],
       [2.75863249e-01],
       [8.08392236e-02],
       [9.52317128e-01],
       [4.79620083e-01],
       [3.68546550e-01],
       [4.01890131e-01],
       [6.85172560e-03],
       [8.77817032e-02],
       [2.70042504e-01],
       [4.45519658e-03],
       [5.68301403e-01],


8. Compute the aleatoric uncertainty for each classification test example for the Gaussian VI model.

In [105]:
y_test_aleatoric = np.mean(y_test_entropies, axis=0)  # Computer aleatoric uncertainty

In [106]:
y_test_aleatoric

array([[3.27198119e-01],
       [3.01403522e-01],
       [1.21172651e-01],
       [5.48228289e-01],
       [1.15010212e-01],
       [2.50048931e-01],
       [2.39390461e-01],
       [8.41101074e-02],
       [3.70217506e-01],
       [3.57157950e-01],
       [5.76022728e-01],
       [1.93429642e-01],
       [4.73198811e-02],
       [1.77175271e-01],
       [2.96106646e-01],
       [2.94523343e-02],
       [7.66586022e-02],
       [1.97576066e-01],
       [8.03880484e-02],
       [3.52510450e-01],
       [2.67301139e-01],
       [3.97112337e-01],
       [2.92964743e-01],
       [4.64661480e-01],
       [3.42508003e-01],
       [3.75091966e-01],
       [6.24889058e-01],
       [6.29273504e-01],
       [3.29139819e-01],
       [5.59295788e-01],
       [2.46539158e-01],
       [1.47132298e-01],
       [6.58727562e-01],
       [5.71332859e-01],
       [6.13159008e-01],
       [3.02724859e-02],
       [2.73873975e-01],
       [4.87991850e-01],
       [2.32257654e-02],
       [5.43547022e-01],


9. Compute the epistemic uncertainty for each classification test example for the Gaussian VI model.

In [107]:
y_test_uncertainty = np.array(
    [-(x * np.log(x) + (1 - x) * np.log(1 - x)) for x in y_test_probs_avg])  # Compute the total uncertainty
y_test_epistemic = y_test_uncertainty - y_test_aleatoric  # Compute the epistemic uncertainty

In [108]:
y_test_epistemic

array([[1.98398583e-02],
       [2.69652945e-02],
       [2.60890288e-02],
       [7.97709999e-02],
       [2.44521343e-02],
       [7.74194401e-02],
       [1.00712463e-01],
       [1.21445298e-02],
       [1.42075343e-02],
       [5.37282497e-02],
       [1.50579046e-02],
       [1.25650477e-02],
       [6.68720414e-03],
       [2.10153297e-02],
       [6.24392472e-02],
       [6.23876358e-03],
       [3.47839988e-02],
       [3.54997408e-02],
       [9.19920236e-03],
       [1.64026174e-02],
       [1.10408398e-02],
       [1.69891769e-02],
       [9.87010533e-02],
       [2.01692344e-02],
       [3.84901278e-02],
       [1.63409718e-01],
       [5.12184950e-02],
       [4.85023709e-02],
       [1.40541786e-01],
       [2.97079552e-02],
       [3.42751199e-02],
       [4.45029406e-02],
       [3.35887059e-02],
       [8.68447380e-02],
       [6.06116017e-02],
       [1.06996070e-02],
       [2.35012521e-02],
       [9.53092609e-02],
       [5.33851891e-03],
       [1.40240759e-01],


In [109]:
from sklearn.metrics import classification_report

for data, label in test_dataloader:
    print(classification_report(label.flatten(), y_test_probs_avg.round().flatten()))

              precision    recall  f1-score   support

           0       0.92      0.96      0.94       607
           1       0.85      0.75      0.80       193

    accuracy                           0.91       800
   macro avg       0.89      0.85      0.87       800
weighted avg       0.91      0.91      0.91       800

